# Scoring — The 5-Component Quad Scorer

This notebook walks through the scoring function that `detect_grid`'s Step 1 uses to pick the best candidate quadrilateral from a `cv2.RETR_TREE` contour set. It's a five-term weighted sum of three classical quad primitives plus two structure-aware signals that check the *interior* of each candidate for grid-like content.

**Prerequisites.** Full dev requirements and GT images (see [`01_detection.ipynb`](01_detection.ipynb)).

**Related reading**

- [`01_detection.ipynb`](01_detection.ipynb) — the overall fallback chain
- `../app/core/extraction.py:336` — `score_grid_structure` implementation
- `../app/core/extraction.py:411` — `score_cell_count` implementation
- `../app/core/extraction.py:468` — `_find_best_quad_structured` (Step 1 orchestration)

In [ ]:
import sys
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from app.core.extraction import (
    score_grid_structure,
    score_cell_count,
    _preprocess,
    order_points,
)

IMAGES_DIR = PROJECT_ROOT / "Examples" / "Ground Example"


## 1. The problem

With `cv2.RETR_TREE` instead of `RETR_EXTERNAL`, Step 1 of the detection chain sees dozens of quad candidates per image — the Sudoku grid, the crossword sitting next to it, the enclosing page border, ad blocks, and noise. Area + centeredness alone can't rank these: the crossword block is often both larger and more centered than the Sudoku grid on the page.

The scoring function has to answer: **does this candidate's interior look like a 9×9 grid?**

The final weighted sum is:

```
score  =  0.20 · area_norm        ← classical: size relative to largest candidate
       +  0.20 · squareness       ← classical: aspect ratio of bounding rect
       +  0.10 · centeredness     ← classical: 1 − (distance from image center)
       +  0.30 · grid_structure   ← NEW: line-pattern check on warped interior
       +  0.20 · cell_count       ← NEW: connected-component count on warped interior
```

The two new terms get the biggest weights. This notebook walks through each term in isolation on image `_33_` (the crossword-nested Sudoku that single-pass detectors fail on).


In [ ]:
# Load _33_ and find all TREE contour candidates
img = cv2.imread(str(IMAGES_DIR / "_33_3803709.jpeg"))
rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
thresh = _preprocess(gray)

contours, _ = cv2.findContours(thresh, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
h, w = img.shape[:2]
image_area = h * w

candidates = []
for c in contours:
    area = cv2.contourArea(c)
    if area < 0.01 * image_area or area > 0.99 * image_area:
        continue
    peri = cv2.arcLength(c, True)
    approx = cv2.approxPolyDP(c, 0.02 * peri, True)
    if len(approx) == 4 and cv2.isContourConvex(approx):
        candidates.append(approx)

print(f"Found {len(candidates)} 4-vertex convex candidates inside {image_area} pixels")

overlay = rgb.copy()
for i, quad in enumerate(candidates):
    colour = tuple(int(c) for c in np.random.default_rng(i).integers(64, 255, size=3))
    cv2.polylines(overlay, [quad.reshape(-1, 1, 2)], isClosed=True, color=colour, thickness=3)

plt.figure(figsize=(8, 8))
plt.imshow(overlay); plt.title(f"{len(candidates)} convex quad candidates (TREE hierarchy)"); plt.axis("off")
plt.show()


## 2. The three classical primitives

`area_norm`, `squareness`, and `centeredness` are standard quad-scoring signals — cheap, unit-free, easy to reason about. None of them can distinguish a crossword block from a Sudoku grid, but together they cull obviously-bad candidates (too small, too rectangular, stuck in the corner).

- **`area_norm`** = `area / max_candidate_area`  — normalised to the largest candidate to make it unit-free.
- **`squareness`** = `min(w, h) / max(w, h)` from `cv2.boundingRect` — penalises elongated or tall-thin quads.
- **`centeredness`** = `1 − (distance_to_image_center / max_possible_distance)` — prefers candidates in the middle of the frame.


In [ ]:
img_center = np.array([w / 2.0, h / 2.0])
max_dist = np.sqrt((w / 2.0) ** 2 + (h / 2.0) ** 2)
max_area = max(cv2.contourArea(q) for q in candidates) if candidates else 1.0

def classical_scores(quad):
    pts = quad.reshape(-1, 2).astype(np.float32)
    area = cv2.contourArea(quad)
    area_norm = area / max_area if max_area > 0 else 0.0
    x, y, bw, bh = cv2.boundingRect(quad)
    squareness = min(bw, bh) / max(bw, bh) if max(bw, bh) > 0 else 0.0
    centroid = pts.mean(axis=0)
    dist = np.linalg.norm(centroid - img_center)
    centeredness = 1.0 - (dist / max_dist) if max_dist > 0 else 1.0
    return area_norm, squareness, centeredness

print(f"{'idx':>4}  {'area_norm':>10}  {'squareness':>11}  {'centeredness':>13}")
for i, quad in enumerate(candidates):
    a, s, c = classical_scores(quad)
    print(f"{i:>4}  {a:>10.3f}  {s:>11.3f}  {c:>13.3f}")


## 3. `grid_structure` — does the interior look like a 9×9 grid?

The decisive signal. For each candidate, warp its interior to a fixed 200×200 square and ask: *are there ~10 evenly-spaced lines per axis covering the full span?*

**Why 10 lines.** A 9×9 Sudoku grid has 10 line positions per axis: 8 interior dividers plus 2 outer borders. Anything fewer is a degenerate grid (partial), anything more is an over-segmented box (contains both the grid and something else).

The computation:

1. `cv2.getPerspectiveTransform` warps the candidate to a 200×200 square.
2. Morphological opening with a `(size//4, 1)` kernel extracts horizontal lines; same with `(1, size//4)` for vertical lines.
3. Project each line image to a 1-D profile (`sum` along the opposite axis).
4. Find peaks above 30% of the max with min spacing = `size/15`.
5. Score three things multiplicatively (all three must be good):
   - **count accuracy**: `max(0, 1 − |n_peaks − 10| / 10)`
   - **spacing regularity**: `max(0, 1 − std(gaps) / expected_gap)`  where expected_gap ≈ 22 px in the warp
   - **coverage**: `(last_peak − first_peak) / 200` — penalises peaks clustered in a small vertical band (e.g., header text above the grid)
6. Final: `grid_structure = (h_score + v_score) / 2`

The multiplicative combination means a candidate has to be good on *all three* dimensions to score well. A crossword block might have close to 10 horizontal lines (by coincidence), but they won't be evenly spaced OR cover the full vertical span.


In [ ]:
print(f"{'idx':>4}  {'struct':>7}  {'n_h':>4}  {'n_v':>4}  {'h_cov':>6}  {'v_cov':>6}")
for i, quad in enumerate(candidates):
    q = quad.reshape(-1, 2).astype(np.float32)
    struct, n_h, n_v, h_cov, v_cov = score_grid_structure(img, q)
    print(f"{i:>4}  {struct:>7.3f}  {n_h:>4}  {n_v:>4}  {h_cov:>6.3f}  {v_cov:>6.3f}")


## 4. `cell_count` — does the interior contain ~81 cells?

A second independent check on the same warped candidate. Inverting the thresholded warp gives white cells on a dark line background; `cv2.connectedComponentsWithStats` returns one component per cell. Filter to components with area in `[0.2, 3.0] × expected_cell_area` where `expected_cell_area = (200 / 9)² ≈ 494 px²` in the warped space.

Score multiplicatively:

- **closeness to 81**: `max(0, 1 − |cell_count − 81| / 81)`
- **size consistency**: `max(0, 1 − coefficient_of_variation_of_areas)` — penalises mixed-size cells
- Final: `cell_count_score = closeness × consistency`

A candidate with 81 components but wildly inconsistent sizes (e.g., a crossword with irregular word boxes) scores low on consistency and loses.


In [ ]:
print(f"{'idx':>4}  {'cells':>6}  {'count':>6}  {'median_area':>12}")
for i, quad in enumerate(candidates):
    q = quad.reshape(-1, 2).astype(np.float32)
    cells_score, n_cells, median_area = score_cell_count(img, q)
    print(f"{i:>4}  {cells_score:>6.3f}  {n_cells:>6}  {median_area:>12.1f}")


## 5. The weighted sum and the `_33_` showdown

Combining all 5 components on every candidate gives a single ranking. The Sudoku grid candidate should win over the crossword candidate — even though the crossword is larger and more centered — because the two structure-aware signals crush the crossword's score.


In [ ]:
def full_score(quad):
    a, s, c = classical_scores(quad)
    q = quad.reshape(-1, 2).astype(np.float32)
    struct, *_ = score_grid_structure(img, q)
    cells, *_ = score_cell_count(img, q)
    total = 0.2*a + 0.2*s + 0.1*c + 0.3*struct + 0.2*cells
    return total, (a, s, c, struct, cells)

ranking = sorted(
    [(i, *full_score(q)) for i, q in enumerate(candidates)],
    key=lambda r: r[1],
    reverse=True,
)

print(f"{'idx':>4}  {'score':>6}  {'area':>6}  {'sq':>6}  {'cent':>6}  {'struct':>7}  {'cells':>6}")
for i, total, (a, s, c, struct, cells) in ranking[:10]:
    print(f"{i:>4}  {total:>6.3f}  {a:>6.3f}  {s:>6.3f}  {c:>6.3f}  {struct:>7.3f}  {cells:>6.3f}")

# Highlight the top-scoring candidate
best_idx = ranking[0][0]
winner = candidates[best_idx]
overlay = rgb.copy()
cv2.polylines(overlay, [winner.reshape(-1, 1, 2)], isClosed=True, color=(0, 255, 0), thickness=4)
plt.figure(figsize=(8, 8))
plt.imshow(overlay); plt.title(f"Winner (candidate idx {best_idx}, score {ranking[0][1]:.3f})"); plt.axis("off")
plt.show()


## 6. Why these weights (0.2 / 0.2 / 0.1 / 0.3 / 0.2)

The weights were tuned on the 38-image ground-truth set. Starting from the classical weights `0.4 / 0.3 / 0.3` and adding the structure signals with equal weight degraded performance on `_33_` because area still dominated. Shifting weight off `area_norm` (0.4 → 0.2), halving `centeredness` (0.3 → 0.1), and giving `grid_structure` the single largest weight (0.3) produced the 34/38 detection rate on the GT set without regressing on any of the easy images. The weights aren't load-bearing at the second decimal place — small perturbations give the same rankings — but the ordering matters: `grid_structure > cell_count ≈ area_norm ≈ squareness > centeredness`.

## 7. References

- `app/core/extraction.py:468` — `_find_best_quad_structured` (the Step 1 scorer)
- `app/core/extraction.py:336` — `score_grid_structure`
- `app/core/extraction.py:411` — `score_cell_count`
- `evaluation/evaluate_detection.py` — production detector benchmark (34/38, median corner error 1.6 px, median IoU 0.99)
- [`01_detection.ipynb`](01_detection.ipynb) — how this scorer fits into the 4-step chain